# Pipeline xử lý dữ liệu Creator

**Đầu vào:** `results_success.jsonl`  
**Đầu ra:** `creators_processed.csv`

| Bước | Notebook gốc | Mô tả |
|------|-------------|-------|
| 1 | `collect.ipynb` | Parse JSONL → gom creator + video, deduplicate |
| 2 | `expand.ipynb` | Lọc thiếu, mở rộng cột phân phối |
| 3 | `feature_engineering.ipynb` | Tính đặc trưng cấp creator |
| 4 | `processing.ipynb` | Xóa cột thừa, điền giá trị thiếu |

## 0. Import thư viện

In [1]:
import json
import ast
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

INPUT_FILE  = 'results_success.jsonl'
OUTPUT_FILE = 'creators_processed_2.csv'

## Bước 1 — Thu thập dữ liệu

Đọc từng dòng JSONL, trích xuất thông tin creator (profile, credit score, 
thống kê hiệu suất, phân phối audience) và danh sách video gần đây.  
Dùng `dict` để **deduplicate** creator theo `aioCreatorID` và video theo `itemID`.
Mỗi dòng cuối cùng trong DataFrame tương ứng với **1 video** của 1 creator.

In [4]:
def _safe_first(lst):
    """Lấy phần tử đầu tiên của list một cách an toàn."""
    return lst[0] if isinstance(lst, list) and lst else {}


def collect_creators(input_file: str) -> pd.DataFrame:
    """
    Parse file JSONL, gom thông tin creator + video.
    Trả về DataFrame với mỗi dòng là 1 video.
    """
    creators_master: dict = {}

    def _update(target, src, mapping):
        """Chỉ ghi nếu key chưa có VÀ giá trị không rỗng."""
        for key, path in mapping.items():
            val = src.get(path)
            if val is not None and val != '' and key not in target:
                target[key] = val

    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            try:
                line_data = json.loads(line)
            except json.JSONDecodeError:
                continue

            for req in line_data.get('requests', []):
                body_raw = req.get('body_raw')
                if not body_raw:
                    continue
                try:
                    creators = json.loads(body_raw).get('creators', [])
                except (json.JSONDecodeError, AttributeError):
                    continue

                for c in creators:
                    cid = c.get('aioCreatorID')
                    if not cid:
                        continue

                    if cid not in creators_master:
                        creators_master[cid] = {'info': {}, 'videos': {}}

                    master   = creators_master[cid]
                    tt_info  = c.get('creatorTTInfo', {})
                    es_data  = c.get('esData', {})
                    price    = es_data.get('price', {})
                    cs       = c.get('creditScore', {})
                    ri       = c.get('riskInfo', {})
                    cvs      = c.get('creatorValueStat', {})
                    stat     = c.get('statisticData', {})
                    op       = stat.get('overallPerformance', {})
                    fdd      = stat.get('followerDistriData', {})

                    if isinstance(op, list):
                        op = _safe_first(op)

                    _update(master['info'], op, {
                        'avgSixSecondsViewsBenchMarkViews': 'avgSixSecondsViewsBenchMarkViews',
                        'avgSixSecondsViewsRate':           'avgSixSecondsViewsRate',
                        'avgSixSecondsViewsRateRank':       'avgSixSecondsViewsRateRank',
                        'engagementRateBenchMark':          'engagementRateBenchMark',
                        'engagementRateRank':               'engagementRateRank',
                        'followerTier':                     'followerTier',
                        'followersGrowthRate':              'followersGrowthRate',
                        'followersGrowthRateRank':          'followersGrowthRateRank',
                        'medianBenchMarkViews':             'medianBenchMarkViews',
                        'medianViewsRank':                  'medianViewsRank',
                        'videoCompleteRate':                'videoCompleteRate',
                        'videoCompleteRateRank':            'videoCompleteRateRank',
                        'engagementRate':                   'engagementRate',
                        'medianViews':                      'medianViews',
                    })
                    _update(master['info'], tt_info, {
                        'tiktokUID':   'ttUID',
                        'nickname':    'nickName',
                        'handleName':  'handleName',
                        'followerCnt': 'followerCnt',
                    })
                    _update(master['info'], c,     {'displayType':       'displayType'})
                    _update(master['info'], price, {
                        'recommendRate100k': 'recommendRate100k',
                        'startingRate100k':  'startingRate100k',
                    })
                    _update(master['info'], cs, {
                        'currentScore':    'currentScore',
                        'currentTier':     'currentTier',
                        'scoreLowerLimit': 'scoreLowerLimit',
                        'scoreUpperLimit': 'scoreUpperLimit',
                    })
                    _update(master['info'], cvs, {
                        'broadcastingScore':   'broadcastingScore',
                        'collaborationScore':  'collaborationScore',
                        'commercialScore':     'commercialScore',
                        'comprehensiveScore':  'comprehensiveScore',
                    })

                    if c.get('contentLabels'):
                        master['info']['contentLabels'] = '|'.join(
                            lbl.get('labelName', '') for lbl in c['contentLabels']
                        )

                    dil = ri.get('disciplineInfoList')
                    if dil:
                        d = _safe_first(dil)
                        master['info']['disciplineStatus'] = d.get('disciplineStatus')
                        master['info']['disciplineType']   = d.get('disciplineType')

                    rel = ri.get('riskEventInfoList')
                    if rel:
                        r = _safe_first(rel)
                        master['info']['riskEventStatus'] = r.get('riskEventStatus')
                        master['info']['riskEventType']   = r.get('riskEventType')

                    for stat_key in ['active', 'age', 'deviceBrand', 'gender', 'region']:
                        if fdd.get(stat_key):
                            master['info'][f'{stat_key}_dist'] = str(fdd[stat_key])

                    for v in c.get('recentItems', []):
                        vid = v.get('itemID')
                        if vid:
                            master['videos'][vid] = {
                                'video_itemID':     vid,
                                'video_title':      v.get('title'),
                                'video_views':      v.get('views'),
                                'video_heart':      v.get('heart'),
                                'video_comment':    v.get('comment'),
                                'video_share':      v.get('share'),
                                'video_createTime': v.get('createTime'),
                            }

    final_rows = []
    for cid, data in creators_master.items():
        base = {'aioCreatorID': cid, **data['info']}
        if data['videos']:
            for v_info in data['videos'].values():
                final_rows.append({**base, **v_info})
        else:
            final_rows.append(base)

    df = pd.DataFrame(final_rows)
    print(f'[collect] {df["aioCreatorID"].nunique():,} creators | {len(df):,} dòng video')
    return df


# df = collect_creators(INPUT_FILE)

NameError: name 'df' is not defined

Entertainment Creator of the Year - Nhà sáng tạo nội dung Giải trí của năm
Sport Creator of the Year - Nhà sáng tạo nội dung Thể thao của năm
Beauty Creator of the Year - Nhà sáng tạo nội dung Làm đẹp của năm
Food Creator of the Year - Nhà sáng tạo nội dung Ẩm thực của năm
Education Creator of the Year - Nhà sáng tạo nội dung Giáo dục của năm

In [32]:
# target_labels = {
#     'Nail Art & Care',
#     'Beauty Tutorials & Tips',
#     'Hair Design & Care'
# }

# df['contentLabels_list'] = df['contentLabels'].str.split('|')

# df = df[
#     df['contentLabels_list'].apply(lambda x: any(label in target_labels for label in x))
# ]

In [33]:
# target_labels = {
#     'Movies & TV',
#     'Theater & Stage',
#     'Music',
#     'Dance',
#     'Animation & Cosplay',
#     'Entertainment News',
#     'Supernatural & Horror'
# }

# df['contentLabels_list'] = df['contentLabels'].str.split('|')

# df = df[
#     df['contentLabels_list'].apply(lambda x: any(label in target_labels for label in x))
# ]

In [34]:
# # Fitness
# # Health & Wellness
# # Fishing, Hunting, & Camping
# # Traditional Sports
# # Extreme Sports
# # Sports News

# Food Display & Reviews
# Beverages & Production
# Mukbang & Food Tasting
# Cooking & Recipes
# Restaurant Exploration



In [35]:
# df.shape

## Bước 2 — Mở rộng cột phân phối

- Thay thế các chuỗi rỗng / `'null'` bằng `NaN`.
- Bỏ creator thiếu `currentTier` (chưa được đánh giá) hoặc thiếu `video_createTime`.
- Parse các cột `*_dist` (dạng chuỗi JSON list) thành các cột số riêng biệt,
  mỗi cột tương ứng với một phân khúc audience (ví dụ `age_18-24`, `gender_female`…).

In [36]:
# # Chuẩn hóa giá trị thiếu
# MISSING_TOKENS = ['', ' ', 'na', 'Na', 'NA', 'null', 'Null', 'None']
# df = df.replace(MISSING_TOKENS, np.nan)

# # Bỏ creator không có đủ thông tin cốt lõi
# df = df.dropna(subset=['video_createTime'])
# print(f'[expand] Sau khi lọc: {df["aioCreatorID"].nunique():,} creators | {len(df):,} dòng')


# def expand_dist_column(series: pd.Series, key_name: str) -> pd.DataFrame:
#     """
#     Parse cột phân phối dạng list-of-dict thành DataFrame.
#     Mỗi key_name trở thành 1 cột với giá trị là ratio.
#     """
#     def _parse(x):
#         if pd.isna(x) or x in ('', 'null'):
#             return {}
#         if isinstance(x, str):
#             try:
#                 x = ast.literal_eval(x)
#             except Exception:
#                 return {}
#         if not isinstance(x, list):
#             return {}
#         return {
#             item[key_name]: item['ratio']
#             for item in x
#             if isinstance(item, dict)
#             and item.get(key_name) is not None
#             and item.get('ratio') is not None
#         }
#     return series.apply(_parse).apply(pd.Series)


# df_active  = expand_dist_column(df['active_dist'],      'active').add_prefix('active_')
# df_age     = expand_dist_column(df['age_dist'],         'ageInterval').add_prefix('age_')
# df_device  = expand_dist_column(df['deviceBrand_dist'], 'deviceBrand').add_prefix('device_')
# df_gender  = expand_dist_column(df['gender_dist'],      'gender').add_prefix('gender_')
# df_country = expand_dist_column(df['region_dist'],      'country').add_prefix('country_')

# df = pd.concat([df, df_active, df_age, df_device, df_gender, df_country], axis=1)
# print(f'[expand] Số cột sau khi mở rộng: {df.shape[1]}')

In [37]:
# df.to_csv('creator_expand.csv', index=False)